
# Add H5 files to database
## Single molecule tracking pipeline, python version

This script uploads the imaging data (in H5 format) to the storage and updates the database. Metadata will be processed automatically from info added during microscopy session. 

The experiment implies a single sample, all files in one folder; it might contain several FOVs, conditions, channels. To proceed, run the cells below by clicking the 'Play' button on the upper panel or select the cell and press Shift+Enter.


In [ ]:
%load_ext dotenv
%dotenv config.env

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
# Import the dependencies.

import os
import numpy as np
import pandas as pd
import h5py
import uuid
import tifffile

from matplotlib import pyplot as plt
from ipyfilechooser import FileChooser
from pathlib import Path
from datetime import datetime

from utils.set_widgets import im_file_chooser, filter_data
from utils.tiff_writer import create_array, write_tiff, h5_upload_img_allas
from utils.conditions_to_dict import conditions_to_df, h5_file_uuids 
from utils.csv_writer import update_db_img
from utils.set_widgets import experiment_widgets
from utils.conditions_to_dict import get_dict, read_h5_meta
from utils.xml_writer import get_xml_from_h5
from utils.csv_writer import h5_update_db_img



In [ ]:
# Run this cell to chose the directory. 
# Click any file in it and press 'Select'.

path = Path(os.getenv('SMPP_INPUT_DIR'))
fc = FileChooser(path)
fc.filter_pattern = '*.h5'
display(fc)


In [ ]:
# Check file sequence and experiment meta

exp_files = []

for i in Path(fc.selected_path).glob('*.h5'):
    f_dict = {}
    f_dict['path']=str(i)
    f_dict['name']=str(i).split('/')[-1]
    f_dict['led']=h5py.File(i, 'r')['MetaData'].attrs['LEDpower']
    f_dict['start']=h5py.File(i, 'r')['MetaData'].attrs['StartTime']
    f_dict['start_hr']=str(datetime.fromtimestamp(f_dict['start']).time())

    exp_files.append(f_dict)
    # print(i)
    # print(h5py.File(i, 'r')['MetaData'].attrs['LEDpower'], h5py.File(i, 'r')['MetaData'].attrs['StartTime'])



exp_files = sorted(exp_files, key=lambda d: d['start_hr'])
last = sorted(exp_files, key=lambda d: d['start_hr'])[-1]

for i in exp_files:
    print(f"{i['name']}, started: {i['start_hr']}, LED power: {i['led']}")

h5file = h5py.File(Path(fc.selected_path)/last['name'], 'r')
exp_name = '_'.join(last['name'].split('_')[:-1])
conditions = read_h5_meta(h5file)
conditions



In [ ]:
# Create metadata XML 

xml_path = get_xml_from_h5(h5file, conditions, exp_name)


In [ ]:
# Update the database (new experiment)

database = pd.read_csv(os.getenv('SMPP_DATABASE'), sep = '\t', dtype = 'str')
new_data = pd.DataFrame(columns = database.columns)
for i in new_data.columns:
    try:
        new_data[i] = [conditions[i]]
    except KeyError:
        pass
new_data['path'] = xml_path
database = pd.concat([database, new_data], ignore_index = True)
database.to_csv(os.getenv('SMPP_DATABASE'), sep = '\t', index = False)

print('database updated:')
database.tail(5)


In [ ]:
# Upload the files, update the database.

fov_uuid = np.nan

for h5file in exp_files:
    # exp_files is list of dictionaries    

    # upload files to allas
    bucket_n, object_n = h5_upload_img_allas(fc.selected_path, h5file['name'])
    
    if h5file['led'] > 0:

        # assign FOV
        fov_uuid = str(uuid.uuid1()) 

        # get 
        file = h5py.File(Path(fc.selected_path)/h5file['name'], 'r')
        proj = np.mean(file['ImageData'], axis=0)
        wf_fname = h5file['name'].split('.')[0]+'_wf_proj.tif'
        tifffile.imwrite(Path(fc.selected_path)/wf_fname, proj) 
        print(f"widefield for omnipose is {wf_fname}")

    # identifiers for new h5 (fluo or second channel)
    git_hash, file_uuid, experiment_uuid = h5_file_uuids(new_data)
        
    # update the database
    updated = h5_update_db_img(new_data, h5file['name'], git_hash, file_uuid, fov_uuid, bucket_n, object_n)


updated.tail(20)



In [ ]:
# To process a new experiment, press 'Kernel' -> 'Restart Kernel'. 
# Otherwise, you can close the add_imaging_data notebook, or shut down the interactive session.
